In [ ]:
from dataclasses import replace, asdict

import pandas as pd
import matplotlib.pyplot as plt

from pathtracer import Renderer, lattice_preview, display_encode

from stats import pixel_distribution

---
# DATA
Renders the test scene once and lists each output image's size, number type and value range, next to the camera settings.

- `beauty` is scene-referred linear radiance in cd/m^2, not display codes
- `rendercam` is the single source of camera state: later sections read their lattice geometry or exposure from it
- Change the camera or AOVs here and rerun all

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
frame = scene.render(camera=rendercam, aovs=("beauty",))

In [ ]:
stats = {k: dict(shape=v.shape, dtype=v.dtype, min=v.min(), max=v.max(), mean=v.mean()) for k, v in frame.items()}
display(pd.DataFrame(stats).T, pd.Series(asdict(rendercam), name="camera").to_frame())

---
# IMAGE
The render as a screen would show it, with its 8-bit pixel counts and its red, green and blue channels.

- `lattice_preview` draws each pixel as its true cell (hexagon or square), so the hex row offset is visible
- Stats and channels use `display_encode` output: camera exposure, OCIO display transform, dither, then 8-bit quantise
- Statistics therefore describe the displayed image, not scene radiance; dither fills codes a plain quantise would leave empty

In [ ]:
x = frame["beauty"]

plt.imshow(display_encode(lattice_preview(x, rendercam, scale=8), camera=rendercam))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# 8-bit stats of beauty as the viewer displays it: pixel_distribution bins codes 0-255, so it takes display_encode output, not cd/m^2
encoded = display_encode(x, camera=rendercam)

_, stat = pixel_distribution(encoded)

labels = ("shape", "pixels", "min", "max", "least_frequent", "most_frequent", "mean", "std", "median", "nonzero", "empty_bins", "unique_colours")
display(pd.Series((encoded.shape, *stat), index=labels, name="beauty").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=[18, 5])
for c, name in enumerate(("R", "G", "B")):
    ax[c].imshow(encoded[..., c], cmap="gray")
    ax[c].set_title(name)
    ax[c].set_xticks([]), ax[c].set_yticks([])
plt.tight_layout()